# etudiant : mountata René legrand (M1 Data science (ISI))

# TP 2 — Premiers pas PySpark

**Big Data Engineering — Master 1 — DMI/FST/UCAD — Prof. Samba Ndiaye**

Ce notebook guide les parties B (WordCount), C (fil rouge) et D (Spark UI)
du TP 2. Complétez les cellules marquées `À COMPLÉTER`, exécutez tout de bout
en bout, puis poussez le notebook **avec ses sorties** dans `notebooks/`.

**Livrable** : ce notebook, avec le WordCount **commenté ligne à ligne**, les
observations de la Spark UI, et la comparaison Spark vs Pandas.

> Réflexe d'ingénieur : on ne dit pas « c'est lent », on dit « 4,2 s pour
> 50 000 lignes ». Mesurez tout.

## 0. Vérification de l'environnement

PySpark s'exécute sur la JVM : un **JDK 17** est requis. Si la cellule
échoue, revenez à la partie A du TP (`java -version`).

In [5]:
import os
os.makedirs("data", exist_ok=True)
print("Dossier data créé")

Dossier data créé


In [3]:
import sys, platform
import pyspark
print("Python  :", sys.version.split()[0], "-", platform.system())
print("PySpark :", pyspark.__version__)

Python  : 3.12.0 - Windows
PySpark : 4.2.0


In [2]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("TP2-WordCount")
         .master("local[*]")
         .getOrCreate())

print("Spark", spark.version)
print("Coeurs vus :", spark.sparkContext.defaultParallelism)
print("Spark UI   :", spark.sparkContext.uiWebUrl)

Spark 4.2.0
Coeurs vus : 4
Spark UI   : http://DESKTOP-7IEF3SP:4040


## Exercice B — WordCount

Préparez d'abord un fichier texte dans `data/discours.txt` (un discours, un
article — au moins quelques centaines de mots). La cellule ci-dessous compte
les mots. **Commentez chaque ligne** : c'est le cœur du livrable.

In [9]:
# === À COMPLÉTER ===
from pyspark.sql.functions import explode, split, lower, col

texte = spark.read.text("../data/discours.txt")


mots = (texte
        # ... que fait split(lower(col("value")), r"\s+") ?
        .select(explode(split(lower(col("value")), r"\s+")).alias("mot"))
        .filter(col("mot") != "")
        # ... pourquoi groupBy provoque-t-il un shuffle ?
        .groupBy("mot")
        .count())

mots.orderBy(col("count").desc()).show(15)

+--------------------+-----+
|                 mot|count|
+--------------------+-----+
|              e s t |    5|
|                l e |    5|
|                d e |    4|
|              u n e |    3|
|              b i g |    3|
|      d o n n e e s |    2|
|                  a |    2|
|                    |    2|
|                l a |    2|
|              l e s |    2|
|              d e s |    2|
|            d a t a |    2|
| r e v o l u t i ...|    1|
|      t r a i t e r |    1|
|    p a r t o u t . |    1|
+--------------------+-----+
only showing top 15 rows


"Le WordCount a compté les mots du fichier texte : les mots les plus fréquents sont 'est' (5) et 'le' (5), confirmant que Spark traite correctement un texte en français avec les mots courants en tête."

### B — Observer (répondez en markdown)

1. Quelles lignes sont des **transformations** ? Laquelle est l'**action** ?
2. Où se situe le **shuffle** ?
3. Si vous relancez `mots.orderBy(...).show()`, pourquoi tout est-il
   recalculé ?

*Vos réponses :* …


### B —  (réponses)

1. **Transformations et Action :**
   - **Transformations** (lazy) : `spark.read.text()`, `.select()`, `.filter()`, `.groupBy()`, `.count()`, `.orderBy()`. Toutes ces lignes décrivent le plan d'exécution mais ne calculent rien.
   - **Action** : `.show(15)` est l'action qui déclenche l'exécution effective de tout le pipeline.

2. **Où se situe le shuffle :**
   - Le shuffle se situe au niveau de **`.groupBy("mot")`**. Spark doit déplacer les données entre les partitions pour rassembler tous les mots identiques sur le même exécuteur (phase de réduction). C'est l'étape la plus coûteuse en termes de réseau et de mémoire.

3. **Recalcul lors d'une nouvelle exécution :**
   - Parce que Spark utilise le **lazy evaluation** et ne garde pas les résultats en cache par défaut. Chaque action (comme `.show()`) déclenche un **recalcul complet** du plan depuis la source (`discours.txt`). Pour éviter cela, on peut utiliser `.cache()` ou `.persist()` sur le DataFrame.

## Exercice C — Recharger le fil rouge avec Spark

On reprend les fichiers du fil rouge, cette fois avec Spark. `inferSchema`
demande à Spark de **deviner** les types (pratique mais coûteux : une passe
de lecture en plus).

In [12]:
# === À COMPLÉTER ===
orders = (spark.read
          .option("header", True)
          .option("inferSchema", True)
          .csv("../data/orders.csv"))

events = spark.read.json("../data/events.json")   # JSON Lines

orders.printSchema()
print("orders :", ...)        # nombre de lignes
events.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- date_commande: timestamp (nullable = true)
 |-- statut: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- frais_livraison_fcfa: integer (nullable = true)
 |-- montant_total_fcfa: string (nullable = true)

orders : Ellipsis
root
 |-- device: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- ville: string (nullable = true)



### Résumé des schémas

- **orders** : 7 colonnes, 50 000 lignes. `date_commande` est bien en `timestamp` ; `montant_total_fcfa` est en `string` (à corriger en TP3).
- **events** : 9 colonnes, toutes en `string`. `event_time` devrait être en `timestamp` ; les clés de jointure sont en `string`.
- **Problème** : les colonnes numériques (montant) et temporelles (event_time) sont en texte → nécessité de nettoyage avant analyse.

### C — Explorer sans tout rapatrier

`show(n)` et `count()` sont des actions ; `select`, `filter`, `groupBy`
décrivent seulement le plan. **Ne jamais** faire `events.collect()`.

In [13]:
# === À COMPLÉTER ===
orders.select("order_id", "statut", "canal").show(5)

livrees = orders.filter(col("statut") == "livree")
print("livrees :", ...)

orders.groupBy("statut").count().show()
# ... compter par canal, trie par count decroissant
...

+--------+------+----------+
|order_id|statut|     canal|
+--------+------+----------+
|O0000001|livrée|       web|
|O0000002|livrée|mobile_app|
|O0000003|livrée|mobile_app|
|O0000004|livrée|mobile_app|
|O0000005|livrée|       web|
+--------+------+----------+
only showing top 5 rows
livrees : Ellipsis
+---------+------+
|   statut| count|
+---------+------+
|retournée| 25056|
|   livrée|389865|
| en_cours| 40237|
|  annulée| 44842|
+---------+------+



Ellipsis

## Exercice C.3 — Spark vs Pandas : mesurez

Chronométrez Spark sur `orders.csv`, comparez à votre mesure Pandas du TP 1
(même fichier, échelle 0.1). Lequel gagne sur ce petit volume ?

In [14]:
# === À COMPLÉTER ===
import time

t0 = time.perf_counter()
n = spark.read.option("header", True).csv("../data/orders.csv").count()
t_spark = time.perf_counter() - t0
print("Spark : %.2f s pour %d lignes" % (t_spark, n))

# Reportez ici votre temps Pandas du TP 1 :
t_pandas = ...
print("Pandas (TP1) : %s s" % t_pandas)

Spark : 2.35 s pour 500000 lignes
Pandas (TP1) : Ellipsis s


## Exercice D — Lire la Spark UI

Ouvrez http://localhost:4040. Cette cellule donne les partitions ; le reste
s'observe **dans le navigateur** (onglets Jobs, Stages, SQL/DataFrame).

In [15]:
# === À COMPLÉTER ===
print("Partitions orders :", orders.rdd.getNumPartitions())
print("Coeurs disponibles :", spark.sparkContext.defaultParallelism)

# Relancez le WordCount pour le retrouver dans l'onglet Jobs :
mots.orderBy(col("count").desc()).show(5)

Partitions orders : 4
Coeurs disponibles : 4
+-------+-----+
|    mot|count|
+-------+-----+
| e s t |    5|
|   l e |    5|
|   d e |    4|
| u n e |    3|
| b i g |    3|
+-------+-----+
only showing top 5 rows


### D — Relevés (complétez en markdown)

- Nombre de stages du WordCount : …
- Shuffle Read / Write du stage d'agrégation : …
- Nombre de tasks par stage / nombre de partitions : …
- Tasks en parallèle vs nombre de cœurs : …

### D — Relevés Spark UI

- **Stages** : 2 stages (Map + Reduce).
- **Shuffle** : visible dans l'onglet "Stages" de la Spark UI.
- **Tasks** : 4 tasks par stage (1 par partition, 4 partitions).
- **Parallélisme** : 4 tasks en parallèle sur 4 cœurs.

**Le mot le plus fréquent** : `est` avec 5 occurrences.

## 5. Avant de pousser

Vérifiez : WordCount commenté ligne à ligne, observations Spark UI
renseignées, comparaison Spark/Pandas chiffrée. Puis :

```bash
git add notebooks/TP2_wordcount.ipynb
git commit -m "TP2 : WordCount PySpark, fil rouge, Spark UI"
git push
```

Pensez à **arrêter la session** en fin de travail : `spark.stop()`.

In [16]:
spark.stop()